In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold Layer: Daily Sales & Fulfillment KPIs (`gold.daily_sales_fulfillment_kpis`)

# COMMAND ----------

from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_fact = spark.table("silver.fct_orders_fulfillment")

# COMMAND ----------

df_daily_kpis = (
    df_fact
    .filter(F.col("order_date").isNotNull())
    .groupBy("order_date", "category", "usr_city", "fulfillment_status")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_sold"),
        F.round(
            F.sum(
                F.when(F.col("total_amount") > 0, F.col("total_amount"))
                .otherwise(F.col("quantity") * 25.0)
            ), 
            2
        ).alias("gross_revenue"),
        F.countDistinct("usr_id").alias("unique_buyers")
    )
    .withColumn("_transformed_at", F.current_timestamp())
)

# COMMAND ----------

(
    df_daily_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.daily_sales_fulfillment_kpis")
)

spark.sql("OPTIMIZE gold.daily_sales_fulfillment_kpis ZORDER BY (order_date, category)")

print("SUCCESS: gold.daily_sales_fulfillment_kpis registered successfully.")

# COMMAND ----------

display(spark.table("gold.daily_sales_fulfillment_kpis").orderBy(F.col("order_date").desc()).limit(10))